# 07 · Spark MLlib: K-means (area types) and FP-Growth (association rules)

**Run this in Google Colab.** It takes about 20–30 minutes. These two methods need **no labels**.

What it does *(Module 3: clustering and association rule mining on spatial big data)*:
1. **Area summaries:** Spark groups all ~13 million roofs into their **H3 hexagons** (~0.1 km² each)
   and summarises each hexagon: number of roofs, typical roof size, share of roofs seen as built-up, tree cover.
2. **K-means clustering:** groups the hexagons into **area types** (for example dense town, suburb, homestead village).
   It tries 3 to 8 groups and picks the number with the best **silhouette score**.
3. **FP-Growth:** turns each roof into a "basket" of words (roof size, what the satellite sees, tree cover,
   crowding, shape) and finds combinations that often appear together, as rules with **support, confidence and lift**.

Outputs (Google Drive): `data/features/hex_clusters.parquet`, and CSV tables in `results/`.
Everything printed is also saved to `MyDrive/rooftop_solar/logs/07_log.txt`.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the roof features to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/07_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


T_START = time.time()
shutil.copytree(f'{FEATURES}/roof_features_v2', f'{LOCAL}/roof_features_v2', dirs_exist_ok=True)
log(f'Roof features copied to {LOCAL} in {time.time() - T_START:.0f} s')

In [ ]:
# Step 2: install Java 17 and PySpark 4.1 (MLlib is part of PySpark; Sedona is not needed here)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3
log('Java:', java_version())

In [ ]:
# Step 3: start Spark, using all CPU cores and most of the RAM
import psutil
from pyspark.sql import SparkSession

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))
spark = (SparkSession.builder.master('local[*]').appName('rooftop_ml')
         .config('spark.driver.memory', f'{MEM_GB}g')
         .config('spark.sql.shuffle.partitions', str(CORES * 4))
         .getOrCreate())
log(f'Spark {spark.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

In [ ]:
# Step 4: load the roofs (only the columns we need; the shapes are not needed for ML)
t = time.time()
roofs = (spark.read.parquet(f'{LOCAL}/roof_features_v2')
         .select('id', 'district', 'area_m2', 'compactness', 'tree_pct_140m', 'tree_pct_300m', 'roof_seen_as',
                 'low_confidence', 'bldgs_in_hex', 'h3_9', 'kwh_year', 'centre_lon', 'centre_lat')
         .cache())
ROOFS = roofs.count()
roofs.createOrReplaceTempView('roofs')
log(f'Roofs loaded: {ROOFS:,} in {time.time() - t:.0f} s')

In [ ]:
# Step 5: K-means clustering of H3 hexagons into area types
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler

t = time.time()
hexes = spark.sql("""
    SELECT h3_9,
           first(district)                                               AS district,
           count(*)                                                      AS roofs,
           percentile_approx(area_m2, 0.5)                               AS median_roof_m2,
           100 * avg(CAST(roof_seen_as = 'built-up' AS DOUBLE))          AS pct_seen_built_up,
           avg(tree_pct_140m)                                            AS tree_140m,
           avg(tree_pct_300m)                                            AS tree_300m,
           sum(kwh_year) / 1e3                                           AS mwh_per_year,
           avg(centre_lon)                                               AS lon,
           avg(centre_lat)                                               AS lat
    FROM roofs GROUP BY h3_9
""")
hexes = (hexes.withColumn('log_roofs', F.log1p('roofs'))              # log, so a few huge values do not dominate
              .withColumn('log_roof_m2', F.log1p('median_roof_m2'))
              .cache())
N_HEX = hexes.count()
log(f'\nHexagons summarised: {N_HEX:,} in {time.time() - t:.0f} s')

FEATURES_KM = ['log_roofs', 'log_roof_m2', 'pct_seen_built_up', 'tree_140m', 'tree_300m']
prep = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURES_KM, outputCol='raw', handleInvalid='skip'),
    StandardScaler(inputCol='raw', outputCol='features', withMean=True, withStd=True),   # same scale for all
]).fit(hexes)
scaled = prep.transform(hexes).cache()

evaluator = ClusteringEvaluator(featuresCol='features', metricName='silhouette')
k_rows = []
for k in range(3, 9):
    model = KMeans(k=k, seed=42, featuresCol='features', maxIter=40).fit(scaled)
    score = evaluator.evaluate(model.transform(scaled))
    k_rows.append({'k': k, 'silhouette': round(score, 3)})
    log(f'  k = {k}: silhouette {score:.3f}')
K_BEST = max(k_rows, key=lambda r: r['silhouette'])['k']
log(f'Best number of area types (highest silhouette): k = {K_BEST}')
log_table(pd.DataFrame(k_rows), 'kmeans_k_selection.csv')

km = KMeans(k=K_BEST, seed=42, featuresCol='features', maxIter=40).fit(scaled)
clustered = km.transform(scaled).withColumnRenamed('prediction', 'cluster')
clustered.createOrReplaceTempView('hex_clusters')

log('\nAREA TYPES (one row per cluster; Claude will name them from these profiles)')
log_table(spark.sql("""
    SELECT cluster,
           count(*)                                AS hexagons,
           sum(roofs)                              AS roofs,
           round(avg(roofs))                       AS avg_roofs_per_hex,
           round(avg(median_roof_m2), 1)           AS median_roof_m2,
           round(avg(pct_seen_built_up), 1)        AS pct_seen_built_up,
           round(avg(tree_140m), 1)                AS tree_140m,
           round(avg(tree_300m), 1)                AS tree_300m,
           round(100 * sum(mwh_per_year) / sum(sum(mwh_per_year)) OVER (), 1) AS pct_of_kerala_potential
    FROM hex_clusters GROUP BY cluster ORDER BY avg_roofs_per_hex DESC
""").toPandas(), 'kmeans_cluster_profiles.csv')

log('\nAREA TYPES BY DISTRICT (share of hexagons, %)')
by_d = spark.sql("SELECT district, cluster, count(*) AS n FROM hex_clusters GROUP BY district, cluster").toPandas()
by_d = by_d.pivot(index='district', columns='cluster', values='n').fillna(0)
by_d = (100 * by_d.div(by_d.sum(axis=1), axis=0)).round(1).reset_index()
by_d.columns = ['district'] + [f'cluster_{c}' for c in by_d.columns[1:]]
log_table(by_d, 'kmeans_clusters_by_district.csv')

hex_pd = clustered.select('h3_9', 'district', 'cluster', 'roofs', 'median_roof_m2', 'pct_seen_built_up',
                          'tree_140m', 'tree_300m', 'mwh_per_year', 'lon', 'lat').toPandas()
hex_pd.to_parquet(f'{FEATURES}/hex_clusters.parquet', index=False)
log(f'K-means done in {(time.time() - t) / 60:.1f} min; saved {len(hex_pd):,} hexagons to data/features/hex_clusters.parquet')

In [ ]:
# Step 6: FP-Growth - which roof characteristics often appear together?
from pyspark.ml.fpm import FPGrowth

t = time.time()
baskets = spark.sql("""
    SELECT filter(array(
        CASE WHEN area_m2 < 50 THEN 'roof=small (<50 m2)'
             WHEN area_m2 < 120 THEN 'roof=medium (50-120 m2)'
             WHEN area_m2 < 500 THEN 'roof=large (120-500 m2)'
             ELSE 'roof=very large (>500 m2)' END,
        'seen as=' || coalesce(roof_seen_as, 'unknown'),
        CASE WHEN tree_pct_300m < 70 THEN 'trees 300m=low (<70%)'
             WHEN tree_pct_300m < 95 THEN 'trees 300m=medium (70-95%)'
             ELSE 'trees 300m=high (>=95%)' END,
        CASE WHEN bldgs_in_hex < 50 THEN 'area=village (<50 bldgs/hex)'
             WHEN bldgs_in_hex < 150 THEN 'area=suburb (50-150)'
             ELSE 'area=town (>=150)' END,
        CASE WHEN compactness IS NULL THEN NULL
             WHEN compactness >= 0.70 THEN 'shape=regular' ELSE 'shape=irregular' END,
        CASE WHEN low_confidence THEN 'AI shape=low confidence' END
    ), x -> x IS NOT NULL) AS items
    FROM roofs
""").cache()
log(f'\nBaskets built for {baskets.count():,} roofs')

fp = FPGrowth(itemsCol='items', minSupport=0.05, minConfidence=0.6).fit(baskets)
itemsets = fp.freqItemsets.cache()
log(f'Frequent item combinations (in at least 5% of roofs): {itemsets.count():,}')

raw = fp.associationRules.toPandas()


def as_table(df):
    """Rules as readable text columns."""
    return pd.DataFrame({
        'if_roof_has': df['antecedent'].apply(' + '.join),
        'then_it_has': df['consequent'].apply(' + '.join),
        'support': df['support'].round(3),
        'confidence': df['confidence'].round(3),
        'lift': df['lift'].round(2),
    }).sort_values('lift', ascending=False)


# Many rules are longer copies of a shorter rule with almost the same confidence; keep the shortest one
raw = raw.assign(n_items=raw['antecedent'].apply(len)).sort_values(['n_items', 'confidence'], ascending=[True, False])
kept, keep_rows = [], []
for row in raw.itertuples():
    a, c = set(row.antecedent), tuple(row.consequent)
    if any(kc == c and ka <= a and row.confidence - kconf < 0.02 for ka, kc, kconf in kept):
        continue
    kept.append((a, c, row.confidence))
    keep_rows.append(row.Index)
simple = raw.loc[keep_rows]

log(f'Rules found (confidence >= 60%): {len(raw):,} | after removing near-copies: {len(simple):,} '
    f'| FP-Growth took {(time.time() - t) / 60:.1f} min')
as_table(raw).to_csv(f'{RESULTS}/fpgrowth_rules_all.csv', index=False)
rules = as_table(simple)
rules.to_csv(f'{RESULTS}/fpgrowth_rules.csv', index=False)
log('\nTOP 20 RULES BY LIFT (lift > 1 = the items appear together more often than by chance)')
log_table(rules.head(20))

In [ ]:
# Step 7: done
log(f'\nTotal time {(time.time() - T_START) / 60:.1f} min')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/07_log.txt`.